In [2]:
!pip install transformers

import pandas as pd
import torch
import torch.nn as nn

from transformers import RobertaTokenizer, RobertaModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import classification_report

In [3]:
import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/multimodal_train.tsv', sep='\t')
val_df = pd.read_csv('/content/drive/MyDrive/multimodal_validate.tsv', sep='\t')
test_df = pd.read_csv('/content/drive/MyDrive/multimodal_test_public.tsv', sep='\t')

In [4]:
train_df.columns

Index(['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id',
       'image_url', 'linked_submission_id', 'num_comments', 'score',
       'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label',
       '6_way_label'],
      dtype='object')

In [17]:
train_df = train_df.sample(3000, random_state=42)
val_df = val_df.sample(800, random_state=42)

In [6]:
train_df.head()

,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
91547,crankbait_XL,feeling lucky,1.353970e+09,NaN,True,c7773wn,http://i.imgur.com/Zu6Sx.jpg,13s0vj,NaN,1,psbattle_artwork,feeling lucky,NaN,0,2,4
424990,ApiContraption,cutouts,1.397487e+09,NaN,True,cgs3w93,https://31.media.tumblr.com/d7100866f676a6a376...,2309r1,NaN,1,psbattle_artwork,cutouts,NaN,0,2,4
495536,FerRod05,my ceiling looks like an sd card,1.565055e+09,i.redd.it,True,cmk4af,https://preview.redd.it/zqg8c8fteqe31.jpg?widt...,NaN,3.0,40,pareidolia,My ceiling looks like an sd card,0.86,0,2,2
200727,BlueScreen,join the raaf,1.315071e+09,i.imgur.com,True,k3n1m,https://external-preview.redd.it/q9DNAI6S1OC2v...,NaN,0.0,8,propagandaposters,Join the R.A.A.F.,0.91,0,1,5
186353,vrun1,hangover,1.426882e+09,NaN,True,cplcw5p,http://i.imgur.com/wLCYVSv.jpg,2zp1kl,NaN,28,psbattle_artwork,Hangover?,NaN,0,2,4


In [7]:
train_df = train_df[['title', '2_way_label']]
val_df = val_df[['title', '2_way_label']]
test_df = test_df[['title', '2_way_label']]

In [8]:
train_df = train_df.dropna()
val_df = val_df.dropna()
test_df = test_df.dropna()

train_df['title'] = train_df['title'].str.lower()
val_df['title'] = val_df['title'].str.lower()
test_df['title'] = test_df['title'].str.lower()

In [9]:
train_texts = train_df['title']
val_texts = val_df['title']
test_texts = test_df['title']

train_labels = train_df['2_way_label']
val_labels = val_df['2_way_label']
test_labels = test_df['2_way_label']

In [18]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=48
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=48
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=48
)

In [11]:
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [12]:
train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

In [13]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained('roberta-base')

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.projection = nn.Linear(512, 768)

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):

        roberta_out = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = roberta_out.last_hidden_state

        lstm_out, _ = self.lstm(sequence_output)

        lstm_feat = torch.mean(lstm_out, dim=1)
        roberta_feat = torch.mean(sequence_output, dim=1)

        combined = roberta_feat + self.projection(lstm_feat)

        out = self.dropout(combined)

        return self.fc(out)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)
for param in model.roberta.parameters():
    param.requires_grad = False

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
for epoch in range(2):

    model.train()

    for batch in train_loader:

        optimizer.zero_grad()

        # ✅ INSERT HERE (move to GPU)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} completed")

Epoch 1 completed
Epoch 2 completed


In [21]:
model.eval()

val_preds = []
val_true = []

with torch.no_grad():
    for batch in val_loader:

        outputs = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device)
        )

        preds = torch.argmax(outputs, dim=1)

        val_preds.extend(preds.cpu().tolist())
        val_true.extend(batch['labels'].tolist())

print(classification_report(val_true, val_preds))

              precision    recall  f1-score   support

           0       0.84      0.84      0.84       611
           1       0.75      0.74      0.75       389

    accuracy                           0.81      1000
   macro avg       0.80      0.79      0.79      1000
weighted avg       0.80      0.81      0.80      1000

